## CM4AI KOLF2.1J CRISPRi Perturbation Atlas

Differential gene expression and signature generation for the CM4AI genome-wide CRISPRi perturbation atlas for KOLF2.1J human induced pluripotent stem cells. See additional information about the dataset at the publication and on the site [here](https://y-doctor.github.io/KOLF2.1J_Perturbation_Cell_Atlas/#home).

[Nourreddine S, Doctor Y, Dailamy A, Lee Y-H, Hansen JN, Chinn R et al. A genome-scale CRISPRi perturbation atlas of human induced pluripotent stem cells. Nat Biotechnol 2026; : 1–12](https://www.nature.com/articles/s41587-026-03199-w)

Single-cell count data was downloaded from [Figshare](https://plus.figshare.com/articles/dataset/_b_A_genome-scale_CRISPRi_perturbation_atlas_of_human_induced_pluripotent_stem_cells_b_/27261219)
and pseudo-bulked according to the methods described in Nourreddine et al.

File downloaded from figshare: `KOLF_Pan_Genome_QC_Filtered.h5ad`


  


In [ ]:
from maayanlab_bioinformatics.dge.limma_voom import limma_voom_differential_expression
from maayanlab_bioinformatics.harmonization.ncbi_genes import ncbi_genes_fetch
from maayanlab_bioinformatics.parse.gmt import gmt_write_dict
import numpy as np
import pandas as pd
from pickle import load
from tqdm import tqdm
import anndata as ad
import os
import re
from rpy2.rinterface_lib.embedded import RRuntimeError

In [ ]:
MAX_LENGTH = 1000
PVAL_THRESHOLD = 0.05
GMT_DIR = "gmts/"
OUTPUT_FILE = "cm4ai-lv"

### 1. Load Pseudo-bulked Data

In [ ]:
# Load the data
ko_data= np.load('pseudbulked_data/pseudobulk_ko_counts.npz', allow_pickle=True)
ntc_data = np.load('pseudbulked_data/pseudobulk_ntc_counts.npz', allow_pickle=True)

ko_counts = ko_data['bulk']
ntc_counts = ntc_data['bulk']
ko_genes = ko_data['genes']
ntc_genes = ntc_data['genes']
ntc_code = ntc_data['ntc_code'][0][0]

assert ko_genes.tolist() == ntc_genes.tolist(), "Gene lists do not match between KO and NTC datasets."

In [ ]:
# Load the metadata
with open("ensemble_id_dictionary.pkl", "rb") as f:
    ens_id_dict = load(f)
ens_id_dict = ens_id_dict['Symbol']
with open('target_code_dictionary.pkl', 'rb') as f:
    target_code_dict = load(f)

ens_ids = ko_genes.tolist()

### 2. Limma-Voom DGE

In [ ]:
# results matrices
logfc_mat = np.zeros((len(ens_ids), len(target_code_dict)))
pval_mat = np.zeros((len(ens_ids), len(target_code_dict)))
adjpval_mat = np.zeros((len(ens_ids), len(target_code_dict)))

os.makedirs('limma_voom', exist_ok=True)
for code, target in tqdm(target_code_dict.items()):
    if code in [ntc_code, 1851, 3028]:
        continue
    if os.path.exists(f'limma_voom/lv_results_{target}.parquet'):
        continue
    # Extract the relevant data for the current target
    sample_counts = ko_counts[code]
    control_counts = ntc_counts[code]
    
    sample_labels = [f'{target}_{i}' for i in range(sample_counts.shape[0])]
    control_labels = [f'NTC_{i}' for i in range(control_counts.shape[0])]

    sample_df = pd.DataFrame(sample_counts.T, index=ens_ids, columns=sample_labels)
    control_df = pd.DataFrame(control_counts.T, index=ens_ids, columns=control_labels)

    # Run limma-voom 
    try:
        de_results = limma_voom_differential_expression(
            controls_mat=control_df,
            cases_mat=sample_df,
            voom_design=True
            )
        de_results = de_results.loc[ens_ids,:]

        # update matrices
        logfc_mat[:, code] = de_results['logFC'].values
        pval_mat[:, code] = de_results['P.Value'].values
        adjpval_mat[:, code] = de_results['adj.P.Val'].values

        # Save perturbation results
        de_results.reset_index().to_parquet(f'limma_voom/lv_results_{target}.parquet')
    except RRuntimeError as e:
        print(
            "Failed processing sample:",
            code,
            target,
            str(e)
        )
        continue

# save results matrices
adata = ad.AnnData(X=logfc_mat.T,
                   var=pd.DataFrame(index=ens_ids),
                   obs=pd.DataFrame(index=list(target_code_dict.values())))
adata.layers['pval'] = pval_mat.T
adata.layers['adjpval'] = adjpval_mat.T
adata.write_h5ad('limma_voom_matrices.h5ad')

### 3. Create Gene Sets / Save GMTs

In [ ]:
def _get_protein_coding_genes():
   
    ncbi_gene = pd.read_csv('Mammalia/Homo_sapiens.gene_info.tsv', sep='\t')
    gene_info = ncbi_gene[(ncbi_gene['#tax_id']==9606) & (ncbi_gene['type_of_gene']=='protein-coding')]
    gene_info['Symbol'] = gene_info['Symbol'].str.upper()
    gene_info['dbXrefs'] = gene_info['dbXrefs'].apply(lambda x: re.findall(r"Ensembl:ENSG[0-9]+", x))
    gene_info_ens = gene_info[gene_info['dbXrefs'].apply(len) > 0]
    gene_info_ens['Ensembl'] = gene_info_ens['dbXrefs'].apply(lambda x: x[0].split(':')[1])

    geneids = gene_info_ens.set_index('Ensembl')['Symbol'].to_dict()

In [ ]:
def create_lv_gmt(adata_lv: ad.AnnData, adjpval_threshold: float = 0.05, max_length: int = 1000, out_file: str=None):
    """
    Create LV GMT of up and down genes with adjusted p-value < 0.05 per gene KO
    Output format:
        term \t gene1 \t gene2 \t ... gene250
    """
    logfc_mat = adata_lv.X
    adjpval_mat = adata_lv.layers['adjpval']
    # Create masks for up and down regulated genes
    up_mask = logfc_mat > 0
    down_mask = logfc_mat < 0
    # Subset the adjusted p-value matrix using the masks, setting non-matching values to infinity
    pos_adjpval = np.where(up_mask, adjpval_mat, np.inf)
    neg_adjpval = np.where(down_mask, adjpval_mat, np.inf)
    # create dataframes with gene KOs as columns and ENSG IDs as rows
    pos_df = pd.DataFrame(pos_adjpval.T, index=adata_lv.var_names, columns=adata_lv.obs_names)
    neg_df = pd.DataFrame(neg_adjpval.T, index=adata_lv.var_names, columns=adata_lv.obs_names)
    # subset to protein coding IDs
    geneids = _get_protein_coding_genes()
    pos_df = pos_df[pos_df.index.isin(geneids.keys())]
    neg_df = neg_df[neg_df.index.isin(geneids.keys())]
   
    def get_gene_set(df, col, threshold, cut):
        sorted = df[col].sort_values(ascending=True)
        gene_set = sorted[sorted < threshold].index.map(geneids).values
        if len(gene_set) > cut:
            gene_set = gene_set[:cut]
        return set(gene_set)
    up_gene_sets = {}
    print('Creating up gene sets...')
    zero_sig = []
    for col in tqdm(pos_df.columns):
        term = f"{col}::KOLF2.1J::hiPSCs::144H:::: up"
        gs = get_gene_set(pos_df, col, adjpval_threshold, max_length)
        if len(gs) == 0:
            zero_sig.append(term)
            continue
        up_gene_sets[term] = gs
    print('Creating down gene sets...')
    down_gene_sets = {}
    for col in tqdm(neg_df.columns):
        term = f"{col}::KOLF2.1J::hiPSCs::144H:::: down"
        gs = get_gene_set(neg_df, col, adjpval_threshold, max_length)
        if len(gs) == 0:
            zero_sig.append(term)
            continue
        down_gene_sets[term] = gs
    
    # combined = up_gene_sets | down_gene_sets
    gmt_write_dict(up_gene_sets, f"{GMT_DIR}/{out_file}_up.gmt")
    gmt_write_dict(down_gene_sets, f"{GMT_DIR}/{out_file}_down.gmt")
    print(f"Wrote GMT: {len(up_gene_sets)} up gene sets, {len(down_gene_sets)} down gene sets")
    print(f"No significant genes ({len(zero_sig)}): {zero_sig}")
    return

In [ ]:
create_lv_gmt(adata_lv=adata, adjpval_threshold=PVAL_THRESHOLD, max_length=MAX_LENGTH, out_file = OUTPUT_FILE)